In [6]:
!pip install pandas

import pandas as pd
import zipfile
import os



matches_df = pd.read_csv('ipl_ball_by_ball_2008_2026.csv')


C:\Users\Ch Chetan\AppData\Local\Temp\ipykernel_16896\1237016012.py:9: DtypeWarning: Columns (36: review_by, 37: review_decision) have mixed types. Specify dtype option on import or set low_memory=False.
  matches_df = pd.read_csv('ipl_ball_by_ball_2008_2026.csv')


In [7]:
chennai_team = "Chennai Super Kings"
opponent_teams = ["Royal Challengers Bengaluru", "Kolkata Knight Riders", "Mumbai Indians"]
home_grounds = ["Eden Gardens", "Chinnaswamy", "Chidambaram", "Wankhede"]

match_records = []

for match_id, group in matches_df.groupby("match_id", sort=False):
    teams = pd.unique(group[["batting_team", "bowling_team"]].to_numpy().ravel())
    teams = [team for team in teams if pd.notna(team)]

    if len(teams) < 2:
        continue

    team1, team2 = teams[:2]
    team_runs = group.groupby("batting_team")["total_runs"].sum()
    team1_runs = int(team_runs.get(team1, 0))
    team2_runs = int(team_runs.get(team2, 0))

    if team1_runs > team2_runs:
        winner = team1
    elif team2_runs > team1_runs:
        winner = team2
    else:
        winner = "Tie"

    match_records.append({
        "match_id": match_id,
        "season": group["season"].iloc[0],
        "date": group["date"].iloc[0],
        "venue": group["venue"].iloc[0],
        "team1": team1,
        "team2": team2,
        "team1_runs": team1_runs,
        "team2_runs": team2_runs,
        "winner": winner
    })

match_summary_df = pd.DataFrame(match_records)

is_csk_match = (
    ((match_summary_df["team1"] == chennai_team) & (match_summary_df["team2"].isin(opponent_teams))) |
    ((match_summary_df["team2"] == chennai_team) & (match_summary_df["team1"].isin(opponent_teams)))
)

filtered_df = match_summary_df[is_csk_match].copy()

In [8]:
venue_pattern = "|".join(home_grounds)
filtered_df = filtered_df[filtered_df["venue"].str.contains(venue_pattern, case=False, na=False)]

In [9]:
extracted_data = []

for _, row in filtered_df.iterrows():
    if row["team1"] == chennai_team:
        runs = row.get("team1_runs", 0)
    else:
        runs = row.get("team2_runs", 0)
        
    if row["winner"] == chennai_team:
        win_loss = "Win"
    elif pd.isna(row["winner"]) or row["winner"] == "Tie":
        win_loss = "Tie/No Result"
    else:
        win_loss = "Loss"
    is_home = "Chidambaram" in str(row["venue"])  
        
    extracted_data.append({
        "team": chennai_team,
        "ground": row["venue"],
        "runs": runs,
        "win/loss": win_loss,
        "is_home": is_home
    })

result_df = pd.DataFrame(extracted_data)
display(result_df)

,team,ground,runs,win/loss,is_home
0,Chennai Super Kings,"MA Chidambaram Stadium, Chepauk",208,Win,True
1,Chennai Super Kings,"MA Chidambaram Stadium, Chepauk",152,Win,True
2,Chennai Super Kings,M Chinnaswamy Stadium,178,Win,False
3,Chennai Super Kings,Wankhede Stadium,156,Loss,False
4,Chennai Super Kings,Eden Gardens,55,Loss,False
...,...,...,...,...,...
72,Chennai Super Kings,Eden Gardens,183,Win,False
73,Chennai Super Kings,M Chinnaswamy Stadium,207,Loss,False
74,Chennai Super Kings,"MA Chidambaram Stadium, Chepauk",192,Win,True
75,Chennai Super Kings,Wankhede Stadium,207,Win,False
